In [ ]:
from ABL import LocalizationSession, preprocessing
import os
from pathlib import Path

def _find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'pyproject.toml').exists():
            return str(p)
    return str(Path.cwd())

wrkdir = _find_repo_root()

To run adpative protocol, load E-field model, corex mesh, and stimulator.

In [ ]:
# Locate the bundled example data (E-field set and cortex mesh)
example_model = os.path.join(wrkdir, "examples", "saved", "example_subject", "example_model.pkl")
E, _, cortex = preprocessing.load_example_model(example_model)

# Where to save this session's outputs
save_path = os.path.join(wrkdir, "examples", "saved", "example_subject")

# Specify stimulator with pre-defined hardware constraints
stimulator_path = os.path.join(wrkdir, "examples", "coils", "5coil_example.json")
stimulator = preprocessing.get_stimulator_from_file(stimulator_path, constraint_type='5coil_strain')

# Start localization instance
device = 'cpu'  # or 'cuda' if you have a compatible GPU
S = LocalizationSession(cortex=cortex, save_path=save_path, stimulator=stimulator, efield_set=E, device=device)

# Add source point for simulating responses
source_ind = [2679]
S.generate_source(source_ind)

The localization state is shown on a Dash server at: http://127.0.0.1:8050/.

Probability prior can be set by clicking the probability mesh before localization.

Run the next cell to start real-time localization.

In [ ]:
# Run real-time localization
trial_count = 50

for i in range(S.trial_number,trial_count):

    print(f"Trial: {i+1}")

    # Prepare for a new trial
    S.start_trial()  

    # Optimize the E-field pattern based on the current model
    didt = S.choose_next_stimulus()
    
    S.record_efield_trial(didt)

    # Recorded the stimulation-induced response
    S.simulate_response()

    # Use the measured data to localize the responses
    S.localize()

    # Complete the trial
    S.complete_trial()

Run refined MCMC analysis. Takes a few minutes if device='cuda', 10x more if device='cpu'.

In [ ]:
S.localize(full=True)

In [ ]:
# End session to stop plotting app and free resources
S.end_session()